# Klasifikasi DemogPairs Menggunakan ViT (Emosi dan Umur) & Logistic Regression

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
emotion_features = joblib.load('features/demogpairs_vit-emotion.pkl')
age_features = joblib.load('features/demogpairs_vit-age.pkl')
features = {}
for d in tqdm(data):
    key = d['image_path']
    features[key] = np.array(list(emotion_features[key]) + list(age_features[key]))
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

  0%|          | 0/10800 [00:00<?, ?it/s]

  7%|▋         | 760/10800 [00:00<00:01, 7592.66it/s]

 14%|█▍        | 1520/10800 [00:00<00:01, 7547.36it/s]

 21%|██        | 2286/10800 [00:00<00:01, 7597.02it/s]

 28%|██▊       | 3046/10800 [00:00<00:01, 7567.82it/s]

...

 80%|███████▉  | 8606/10800 [00:01<00:00, 7629.00it/s]

 87%|████████▋ | 9411/10800 [00:01<00:00, 7752.39it/s]

 94%|█████████▍| 10201/10800 [00:01<00:00, 7795.51it/s]

100%|██████████| 10800/10800 [00:01<00:00, 7762.36it/s]

Jumlah fitur per gambar: 1536


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [LogisticRegression(random_state=42)],
        'classifier__C': [0.01, 0.1, 1, 10],
        'classifier__max_iter': [500, 1000],
        'classifier__solver': ['lbfgs', 'saga'],
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

LogisticRegression: 96 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models, 
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix="models/clf_demogpairs_lr_vit-emotion-age_",
    results_path="results/demogpairs_lr_vit-emotion-age_"
)
sorted_results = pd.DataFrame(evaluation_results).sort_values(by="test_accuracy", ascending=False).to_dict("records")
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: LogisticRegression


{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 500, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': None}


Accuracy  : 0.9050925925925926
Precision : 0.905158797415119
Recall    : 0.9050925925925926
F1 Score  : 0.905105871050543
               precision    recall  f1-score   support

Asian_Females     0.8722    0.8722    0.8722       360
  Asian_Males     0.8911    0.8861    0.8886       360
Black_Females     0.8809    0.8833    0.8821       360
  Black_Males     0.9256    0.9333    0.9295       360
White_Females     0.9148    0.9250    0.9199       360
  White_Males     0.9463    0.9306    0.9384       360

     accuracy                         0.9051      2160
    macro avg     0.9052    0.9051    0.9051      2160
 weighted avg     0.9052    0.9051    0.9051      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9574074074074074,0.8722222222222222,0.8722222222222222,0.8722222222222223,360
Asian_Males,0.9629629629629629,0.8910614525139665,0.8861111111111111,0.8885793871866295,360
Black_Females,0.9606481481481481,0.8808864265927978,0.8833333333333333,0.8821081830790568,360
Black_Males,0.9763888888888889,0.9256198347107438,0.9333333333333333,0.9294605809128631,360
White_Females,0.9731481481481481,0.9148351648351648,0.925,0.9198895027624309,360
White_Males,0.9796296296296296,0.9463276836158192,0.9305555555555556,0.938375350140056,360


Confusion matrix saved: images\cm_lr_vit-emotion-age_LogisticRegression.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               314                17                18                 2                 9                 0
         Asian_Males                22               319                 2                 6                 0                11
       Black_Females                15                 1               318                10                16                 0
         Black_Males                 1                10                 9               336                 0                 4
       White_Females                 8                 0                14                 1               333                 4
         White_Males                 0                11                 0                 8                 6               335


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
LogisticRegression,models/clf_demogpairs_lr_vit-emotion-age_LogisticRegression.pkl,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 500, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': None}",0.9050925925925926,0.905105871050543,0.905158797415119,0.9050925925925926,270


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_lr_vit-emotion-age_LogisticRegression.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 7408.0,
 'days': 0,
 'hours': 2,
 'minutes': 3,
 'seconds': 28.0,
 'text': '0 hari 2 jam 3 menit 28.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 85785.0,
 'days': 0,
 'hours': 23,
 'minutes': 49,
 'seconds': 45.0,
 'text': '0 hari 23 jam 49 menit 45.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 500, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': None}",0.9028,0.8889,0.8843,0.8924,0.8947,0.8926,0.8925,0.893,0.8926,29.5151
2,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 1000, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': None}",0.9028,0.8889,0.8843,0.8924,0.8947,0.8926,0.8925,0.893,0.8926,21.3169
3,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 2000, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': None}",0.9028,0.8889,0.8843,0.8924,0.8947,0.8926,0.8925,0.893,0.8926,28.7167
4,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 1000, 'classifier__solver': 'newton-cg', 'pca': None, 'scaler': None}",0.901,0.8889,0.8854,0.8924,0.8947,0.8925,0.8924,0.8929,0.8925,30.2837
...,...,...,...,...,...,...,...,...,...,...,...
267,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 1000, 'classifier__solver': 'saga', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8084,0.805,0.8102,0.809,0.8212,0.8108,0.8107,0.8114,0.8108,20.8421
268,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 2000, 'classifier__solver': 'newton-cg', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8084,0.805,0.8102,0.809,0.8212,0.8108,0.8107,0.8114,0.8108,18.2338
269,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 2000, 'classifier__solver': 'saga', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8084,0.805,0.8102,0.809,0.8212,0.8108,0.8107,0.8114,0.8108,24.8326
270,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 1000, 'classifier__solver': 'newton-cg', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8084,0.805,0.8102,0.809,0.8212,0.8108,0.8107,0.8114,0.8108,20.0389
